In [1]:
from dsa.ms_dsa.Evaluation.binding_interface_eval import BIEvaluation
from dsa.ms_dsa.config import ms_data_relativePath
import warnings
#warnings.filterwarnings("ignore")
binding_interface_eval = BIEvaluation(ms_data_relativePath=ms_data_relativePath, 
                                        distance_cutoff=6.5, 
                                        contact_filter_key="lysine_only",
                                        max_uniprot_ids=500)
# #binding_interface_eval.plot_boxplot()
# for run_index in binding_interface_eval.ms_dsa:
#     binding_interface_eval.plot_dsa_vs_sasa_with_binding_annotation(sasa_type="asa_nz", run_index=run_index)

Processed 0 uniprot_ids
SASA mismatch: 0
example mismatch: None
Processed 100 uniprot_ids
SASA mismatch: 2
example mismatch: A0A2R8Y4L2
Processed 200 uniprot_ids
SASA mismatch: 4
example mismatch: A0A2R8Y4L2


/home/shajain/miniforge3/envs/dsa/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/shajain/miniforge3/envs/dsa/lib/python3.11/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: zero-size array to reduction operation minimum which has no identity

In [ ]:
import matplotlib.pyplot as plt
def histograms(ms_dsa, strict: bool=True, run_index: int=None):
    ms_dsa = ms_dsa[ms_dsa["skipped"] == False]
    if strict:
        inter = ms_dsa[ms_dsa["is_strictly_inter_chain_contact_pdb"]]
    else:
        inter = ms_dsa[ms_dsa["is_inter_chain_contact_pdb"]]
    print(f"Inter: {inter.shape}")
    # intra_pdb = ms_dsa[ms_dsa["is_intra_chain_contact_pdb"]]
    intra_AF = ms_dsa[ms_dsa["is_intra_chain_contact_AF"]]
    #intra = ms_dsa[ms_dsa["is_intra_chain_contact_pdb"]]
    intra = intra_AF
    print(f"Intra: {intra.shape}")
    not_contact = ms_dsa[~ms_dsa["is_inter_chain_contact_pdb"] & ~ms_dsa["is_intra_chain_contact"]]
    plt.hist(inter["MS_DSA"].values, bins=10, label="Inter", alpha=0.25, density=True)
    plt.hist(intra["MS_DSA"].values, bins=10, label="Intra", alpha=0.25, density=True)
    plt.hist(not_contact["MS_DSA"].values, bins=10, label="Not Contact", alpha=0.25, density=True)
    plt.xlabel("MS_DSA")
    plt.ylabel("Frequency")
    plt.title("MS_DSA Distribution on Run {run_index}")
    plt.legend()
    plt.show()

def plot_dsa_vs_sasa_with_binding_annotation(ms_dsa, contact_type, bond_type, sasa_type: str="asa_nz", statistic: str="avg", strict: bool=True, run_index: int=None):
        ms_dsa = ms_dsa[ms_dsa["skipped"] == False]
        sasa_column = f"{sasa_type}_{statistic}"
        if contact_type == "inter_chain":
            inter = ms_dsa[ms_dsa["inter_chain"]]
        else:
            inter = ms_dsa[ms_dsa["intra_chain"]]
        print(f"Inter: {inter.shape}")
        print(f"% of Inter less than 0.9: {len(inter[inter['MS_DSA'] < 0.9]) / len(inter)}")
        #not_inter = self.combined_df[~self.combined_df["is_inter_chain_contacts"]]
        intra_pdb = ms_dsa[ms_dsa["is_intra_chain_contact_pdb"]]
        print(f"Intra: {intra_pdb.shape}")
        print(f"% of Intra less than 0.9: {len(intra_pdb[intra_pdb['MS_DSA'] < 0.9]) / len(intra_pdb)}")
        intra_AF = ms_dsa[ms_dsa["is_intra_chain_contact_AF"]]
        print(f"Intra AF: {intra_AF.shape}")
        print(f"% of Intra AF less than 0.9: {len(intra_AF[intra_AF['MS_DSA'] < 0.9]) / len(intra_AF)}")
        #intra = ms_dsa[ms_dsa["is_intra_chain_contact"]]
        intra = intra_AF
        # print(f"Intra: {intra.shape}")
        # print(f"% of Intra less than 0.9: {len(intra[intra['MS_DSA'] < 0.9]) / len(intra)}")
        not_contact = ms_dsa[~ms_dsa["is_inter_chain_contact_pdb"] & ~ms_dsa["is_intra_chain_contact"]]
        print(f"Not Contact: {not_contact.shape}")
        print(f"% of Not Contact less than 0.9: {len(not_contact[not_contact['MS_DSA'] < 0.9]) / len(not_contact)}")
        plt.scatter(not_contact["MS_DSA"].values, not_contact[sasa_column].values, color="black", label="Not Contact", s=10, alpha=0.5)
        # plt.scatter(not_inter["MS_DSA"].dropna().values, not_inter["RelASA"].dropna().values, color="black", label="Not Inter")
        plt.scatter(intra["MS_DSA"].values, intra[sasa_column].values, color="green", label="Intra", s=10, alpha=0.5)
        # plt.scatter(not_intra["MS_DSA"].dropna().values, not_intra["RelASA"].dropna().values, color="black", label="Not Intra")
        plt.scatter(inter["MS_DSA"].values, inter[sasa_column].values, color="red", label="Inter", s=10, alpha=0.5)
        plt.xlabel("MS_DSA")
        plt.ylabel(sasa_column)
        plt.title(f"MS_DSA vs {sasa_column} on Run {run_index}")
        plt.legend()
        plt.show()



In [2]:
import matplotlib.pyplot as plt
import numpy as np

def _mask(ms_dsa, contact_type, source="pdb", bond_type=None, strict=False):
    contact_col = {
        ("inter_chain", "pdb"): "inter_chain_pdb",
        ("intra_chain", "pdb"): "intra_chain_pdb",
        ("intra_chain", "alphafold"): "intra_chain_alphafold",
    }[(contact_type, source)]
    m = ms_dsa[contact_col].fillna(False).astype(bool)
    if strict and contact_type == "inter_chain":
        m = m & ~ms_dsa["intra_chain_pdb"].fillna(False).astype(bool)
    if bond_type is not None:
        m = m & ms_dsa[f"{bond_type}_{source}"].fillna(False).astype(bool)
    return m

def _frac_lt(df, thresh=0.9):
    if len(df) == 0:
        return np.nan
    return (df["MS_DSA"] < thresh).mean()

def histograms(ms_dsa, source="pdb", bond_type=None, strict=True, run_index=None):
    ms_dsa = ms_dsa[ms_dsa["skipped"] == False]
    inter = ms_dsa[_mask(ms_dsa, "inter_chain", source, bond_type, strict)]
    intra = ms_dsa[_mask(ms_dsa, "intra_chain", "alphafold", bond_type)]
    not_contact = ms_dsa[
        ~_mask(ms_dsa, "inter_chain", "pdb")
        & ~_mask(ms_dsa, "intra_chain", "pdb")
        & ~_mask(ms_dsa, "intra_chain", "alphafold")
    ]
    print(f"Inter: {inter.shape}  Intra: {intra.shape}  Not contact: {not_contact.shape}")
    plt.hist(inter["MS_DSA"].dropna().values, bins=10, label="Inter", alpha=0.25, density=True)
    plt.hist(intra["MS_DSA"].dropna().values, bins=10, label="Intra AF", alpha=0.25, density=True)
    plt.hist(not_contact["MS_DSA"].dropna().values, bins=10, label="Not Contact", alpha=0.25, density=True)
    plt.xlabel("MS_DSA")
    plt.ylabel("Frequency")
    plt.title(f"MS_DSA Distribution on Run {run_index}")
    plt.legend()
    plt.show()

def plot_dsa_vs_sasa_with_binding_annotation(
    ms_dsa,
    contact_type="inter_chain",
    bond_type=None,
    source="pdb",
    sasa_type="asa_nz",
    statistic="avg",
    strict=False,
    run_index=None,
):
    ms_dsa = ms_dsa[ms_dsa["skipped"] == False]
    sasa_column = f"{sasa_type}_{statistic}"

    inter = ms_dsa[_mask(ms_dsa, "inter_chain", "pdb", bond_type, strict)]
    intra = ms_dsa[_mask(ms_dsa, "intra_chain", "alphafold", bond_type)]
    intra_pdb = ms_dsa[_mask(ms_dsa, "intra_chain", "pdb", bond_type)]
    not_contact = ms_dsa[
        ~_mask(ms_dsa, "inter_chain", "pdb")
        & ~_mask(ms_dsa, "intra_chain", "pdb")
        & ~_mask(ms_dsa, "intra_chain", "alphafold")
    ]

    highlight = ms_dsa[_mask(ms_dsa, contact_type, source, bond_type, strict)]

    print(f"Inter PDB: {inter.shape}  ({_frac_lt(inter):.3f} < 0.9)")
    print(f"Intra PDB: {intra_pdb.shape}  ({_frac_lt(intra_pdb):.3f} < 0.9)")
    print(f"Intra AF: {intra.shape}  ({_frac_lt(intra):.3f} < 0.9)")
    print(f"Not contact: {not_contact.shape}  ({_frac_lt(not_contact):.3f} < 0.9)")

    plt.scatter(not_contact["MS_DSA"].values, not_contact[sasa_column].values, color="black", label="Not Contact", s=10, alpha=0.5)
    plt.scatter(intra["MS_DSA"].values, intra[sasa_column].values, color="green", label="Intra AF", s=10, alpha=0.5)
    plt.scatter(highlight["MS_DSA"].values, highlight[sasa_column].values, color="red", label=f"{contact_type}/{source}", s=10, alpha=0.5)
    plt.xlabel("MS_DSA")
    plt.ylabel(sasa_column)
    plt.title(f"MS_DSA vs {sasa_column} on Run {run_index}")
    plt.legend()
    plt.show()

In [1]:
from dsa.ms_dsa.Evaluation.binding_interface_eval import BIEvaluation
ms_dsa_by_run = {}
for run_index in range(5):
    ms_dsa_by_run[run_index] = BIEvaluation.load_processed_ms_dsa(run_index)


In [ ]:
ms_dsa_by_run[0]["asa_nz_avg"].isna().sum()

len(ms_dsa_by_run[1])

10351